First, let's construct our ring.  Each variable $c_{ij}$ will represent the cosine of the angle between the vectors $\mathbf v_i$ and $\mathbf v_j$.  Since we already know a bunch of these are 1 or $\frac{1}{3}$ and the order of $i$ and $j$ don't matter, we only have 9 variables.

The *Weights* were carefully chosen ahead of time to give us a monomial order that gives us the nicest possible Gröbner basis.

In [ ]:
ind = flatten apply(6, i -> apply(4 - i, j -> (i, i + j + 2))) - set{(0,5)}
R = QQ new Array from append(apply(ind, ij -> c_ij), Weights => {0, 1, 1, 1, 1, 1, 0, 1, 1})

Now we construct the Gram matrix.

In [ ]:
getVar = (i,j) -> (
    if i == j then 1
    else if isMember(abs(i - j), {1, 5}) then 1/3
    else if i < j then c_(i,j)
    else c_(j,i));
G = matrix table(6, 6, getVar)

Now we define our ideal from our system of equations.

In [ ]:
I = ideal join(
    apply(subsets(6, 4), ijkl -> det G_ijkl^ijkl),
    apply(6, i -> sum(6, j -> getVar(i,j))))

Next, we find the *minimal primes*, which each correspond to different components of our solution set.

In [ ]:
MP = minimalPrimes I -- Gröbner bases behind the scenes
dim \ MP

Our components consist of a curve, a point, and a point.

We look at the points first.

In [ ]:
vars R % MP#1

Those can't be cosines!

Let's look at the next point.

In [ ]:
sols = vars R % MP#2

That works!

Remember these are cosines of angles between vectors.  We can recover the vectors themselves from the Gram matrix using the *Cholesky decomposition.*

This particular configuration is called the "chair".

In [ ]:
needsPackage "DeterminantalRepresentations"
V = entries (cholesky sub(sub(G, sols), RR))_{0..2}

needsPackage "Plotly"
plot append(prepend({0,0,0}, apply(toList(1..6), i -> sum(i, j -> V#j))), {0,0,0})

Now let's look at the curve.  We start with computing a Gröbner basis.

In [ ]:
J = ideal gens gb MP#0
new VerticalList from J_*

We can rewrite everything in terms of $c_{0,2}$, $c_{2,4}$.  In particular, every solution corresponds to a point on a quartic curve in $\mathbb R^2$.

In [ ]:
S = QQ[x,y]
f = map(S, R, {x,0,0,0,0,0,y,0,0})
g = f J_0
plot g

If we pick an $x$-coordinate of a point on the curve, then we can use this equation to find up to two $y$-coordinates.  Then we can use the Gröbner basis to back-substitute and find everything.

In [ ]:
getVectors = x0 -> (
    y0 = (realPart first roots sub(J_0, c_(0,2) => x0))^QQ;
    z0 = (realPart first roots sub(J_8, {c_(0,2) => x0, c_(2,4) => y0}))^QQ;
    sol = sub(sub(vars R % J, {c_(0,2) => x0, c_(2,4) => y0, c_(1,4) => z0}), RR);
    entries (cholesky sub(sub(G, sol), RR))_{0..2})
getVectors(-1/2)

Let's plot these.  This conformation is known as the "boat".

In [ ]:
plot(apply(101, i -> 5/1000*i - 7/10), x0 -> (
    V := getVectors x0;
    append(prepend({0,0,0}, apply(toList(1..6), i -> sum(i, j -> V#j))), {0,0,0})))